# ExaHyPE2 Tutorials, part 2: the shallow-water equations


We import the required modules for this project.
We always need the 'peano4' module as this is our core project.
Since we are creating an ExaHyPE 2 application, we additionally need to import the 'exahype2' module.

This stuff got lost:

## Two-dimensional Shallow Water Equations
The shallow water equations describe the flow of fluid given the assumption that the horizontal length scale
is much higher than the vertical length scale and are defined by the following formula:

@f$
    \frac{\partial}{\partial t}\left(
    \begin{array}{lr} h \\
        h u_1 \\
        h u_2 \\
        b
        \end{array} \right) +
    \nabla \begin{pmatrix}
        h u_1           & h u_2     \\
        h u_1^2         & h u_1 u_2 \\
        h u_2 u_1       & h u_2^2   \\
        0               & 0
    \end{pmatrix} +
    \begin{pmatrix}
        0                     & 0\\
        h g*\partial_x(b+h)   & 0\\
        0                     & h g*\partial_y(b+h) \\
        0                     & 0
    \end{pmatrix}  = \vec{0}
@f$

With eigenvalues:

@f$
    \left(
    \begin{array}{lr}
        \lambda_1 \\
        \lambda_2 \\
        \lambda_3
    \end{array} \right) =
    \left(
    \begin{array}{lr}
        u + \sqrt{g (h+b)} \\
        u \\
        u - \sqrt{g (h+b)} \\
    \end{array} \right)
@f$

These depend on the fluid height h, velocities in x- and y-directions,
the bathymetry b (the height of the floor underneath the fluid) and the gravity g.

To implement this equation, the following things are needed: the initial condition,
the boundary conditions, the maximal eigenvalue, the flux, the non-conservative term and (optionally) the refinement criterion.

We also have an auxiliary variable since the bathymetry never changes.
We can define this auxiliary variable by setting the parameter for the auxiliary variables in the solver to 1.
These auxiliary variables are accessed in the same way as the regular variables, except that they are not modified by the solver.
Initial- and boundary conditions for these auxiliary variables should be defined, but no flux or non-conservative product can be set,
as they can never change anyway.

### Initial Conditions
We start with the initial conditions: For this, we want the initial water height to be 1.0 everywhere,
the initial velocities to be 0.0 everywhere and the bathymetry should be 0.0 in a radius of 0.40 around
the center of the domain, 0.1 anywhere else.

### Boundary Conditions
Then we can define our boundary conditions. For this simple case, we will simply use reflecting boundary conditions.
This means all values except the momenta will be the same outside of the as inside of it.

### Eigenvalues
Next, we can define the eigenvalues of our system.
This is important because the largest eigenvalue defines the behaviour of the PDE,
and among others the maximum allowable timestep for which the solver remains stable.
Since the maximal eigenvalue is dependent on the velocity,
you will need to take into account the normal direction as this will determine whether you should
use the velocity in the x- or y-direction to compute it. For the gravity g, you can use 9.81 m/s^2.

### Flux
Now, let's define our flux in x- and y-direction.
The directions of the flux are defined by the normal parameter.
This parameter defines through which surface, and therefore in which direction,
the flux is being computed. Here if the normal is 0, we are computing the flux in x-direction
and if the normal is 1, we are computing the flux in y-direction.
For higher dimensions, there would logically be additional directions to implement the flux.

### Non-conservative Term
Finally, we fill in the non-conservative part in the function nonconservativeProduct.
This needs to go into the variable BTimesDeltaQ.
This function has a parameter deltaQ through which you can access the space derivatives of the unknowns and
auxiliary variables in the direction defined by the normal's current value.

### Result
| t=0.0 s                    | t=1.0 s                    |
| :-------------------------:|:-------------------------: |
| ![](SWE_start.png)         | ![](SWE_end.png)           |

The computed height of the fluid according to our computation of the shallow water equations at times t=0.0 s and t=1.0 s.
The fluid has rushed in from the sides to fill the depression at the center.

## Increasing Resolution
If you've completed the implementation, you may have noticed that the resolution is somewhat low.
We're interested in increasing the resolution so we can see more details in the solution.
This means we need to consider our volume size, so in the first step, we will discuss how the volume size is calculated.

This determines the maximal allowable size for volumes in our domain, in essence guaranteeing @f$ max\_volume\_h > volume\_size @f$.
However, since Peano is based upon three-partitioning, it refines the domain into @f$ 3^d @f$ volumes,
where d is the smallest possible integer that satisfies @f$ max\_volume\_h > domain\_size / 3^d @f$.
For our tutorial, we have set a domain size of 2.0 in each direction and @f$ max\_volume\_h = 0.05 @f$.
The first three-partition of 2.0 that is lower than 0.05 is @f$ \frac{2.0}{3^4}\approx0.0247 @f$.
This means that there are a total of 3^4=81 volumes in each dimension. Of these, two are used for the boundary conditions,
which means there are a total of 79 volumes per dimension inside the domain.
In total, that makes 79*79=6241 volumes inside the domain, or (with a parameter @f$ max\_volume\_h = 0.02 @f$)
241*241=58081 volumes in two dimensions, an increase of factor 9.
However, because the smallest admissible timesteps are proportional to the size of the mesh,
the overall computational effort increases by a total factor of about 27.

In short, if one wishes to increase or decrease the mesh size, one can just modify the parameter
@f$ max\_volume\_h @f$, but one should be aware that this comes at a significant cost.
You may have noticed that there is a second parameter @f$ min\_volume\_h @f$ however,
and this parameter logically determines the minimal mesh size.
If this parameter is given a value lower or equal to @f$ max\_volume\_h / 3 @f$ (again, three-partitioning),
adaptive refinement is switched on, this means the volume size need not be the same in the entire domain.


In [ ]:
import sys, os

sys.path.insert(0, os.path.abspath("../../../../python/"))

import peano4, exahype2

Here, we define an ExaHyPE 2 solver, specifically an ADER-DG solver.
It takes as arguments a name, a polynomial order, the unknowns, and the allowable cell sizes for the refinement.

In [ ]:
unknowns = {"h": 1, "hu": 2, "b": 1}
auxiliary_variables = {}
max_h = 1.1 * 2.0 / (3.0 ** 3)
min_h = max_h

my_solver = exahype2.solvers.aderdg.GlobalAdaptiveTimeStep(
    name                  = "SWESolver",
    order                 = 5,
    min_cell_h            = min_h,
    max_cell_h            = max_h,
    time_step_relaxation  = 0.5,
    unknowns              = unknowns,
    auxiliary_variables   = auxiliary_variables
)

my_solver.plot_description = ", ".join(unknowns.keys())

With our solver defined, we can let it know what our equation will look like.
`exahype2.solvers.PDETerms.User_Defined_Implementation` lets the project know that we will manually define the terms in an implementation file.

In [ ]:
my_solver.set_implementation(
    flux = exahype2.solvers.PDETerms.User_Defined_Implementation,
    ncp  = exahype2.solvers.PDETerms.User_Defined_Implementation,
    refinement_criterion = exahype2.solvers.PDETerms.User_Defined_Implementation
)

Here, we define an ExaHyPE 2 project to which our solver will be bound.
This takes a namespace for the C++ code and a name for the executable. We then add our solver to this project and, just for cleanliness, set an output path for our results.

In [ ]:
exahype2_project = exahype2.Project(
    namespace     = ["exahype2", "training", "swe"],
    directory     = ".",
    project_name  = "DamBreak",
    executable    = "DamBreak"
)

exahype2_project.add_solver(my_solver)
exahype2_project.set_output_path("solutions")

Now that we have an ExaHyPE 2 project, let's populate it with some information:
This is essentially the domain our solver will run in.
The project needs to know the domain size, how long the simulation should run,
how often it should plot the solution, and what type of boundary we use.

In [ ]:
exahype2_project.set_global_simulation_parameters(
    dimensions            = 2,
    size                  = [ 2.0,  2.0],
    offset                = [-1.0, -1.0],
    min_end_time          = 0.5,
    max_end_time          = 0.5,
    first_plot_time_stamp = 0.0,
    time_in_between_plots = 0.1,
    periodic_BC           = [False, False],
)

In [ ]:
exahype2_project.set_load_balancing(
    "toolbox::loadbalancing::strategies::SpreadOutOnceGridStagnates",
    "new ::exahype2::LoadBalancingConfiguration()"
)

exahype2_project.set_Peano4_installation("../../../../", peano4.output.CompileMode.Release)
peano4_project = exahype2_project.generate_Peano4_project()
peano4_project.output.makefile.add_CXX_flag("-DExercise=1")
peano4_project.generate()

And finally, we can compile the project.

In this example the implementation is not provided for you, so you will need to look into the implementation file **SWESolver.cpp**, implement the functions and re-compile the project before you can run it. Go do so now!

In [ ]:
peano4_project.build(make=True, make_clean_first=False, throw_away_data_after_build=False)

Once you've implemented the necessary functions and re-compile the project, you can run it.

This is usually done directly in the terminal via `./$executable` but can be done in python as well.

In [ ]:
import subprocess

subprocess.run("./DamBreak", shell = True, executable="/bin/bash")

Finally, once the program has finished running, we can convert the output from the native peano4 format into something that can be visualized.

In [ ]:
import subprocess
subprocess.call(
    ["python3", "../../../../python/peano4/visualisation/render.py", "solutions/solution-SWESolver.peano-patch-file"]
)